# Phase 2 — train and publish on GPU
Use a Colab **A100 40 GB or A100/H100 80 GB GPU runtime**. This notebook downloads the already prepared private dataset bundle, trains both gated stages on `/content`, verifies the merged model, and publishes only the final result to Hugging Face.

In [ ]:
from google.colab import userdata
import shutil, subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/SSusantAchary/coder-SFT.git'
PROJECT_ROOT = Path('/content/coder_SFT')
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'{PROJECT_ROOT} exists but is not a valid coder-SFT checkout')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
WORK_ROOT = Path('/content/qwen35-2b-coder')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage('/content')
print(f'Colab disk: {disk.total / 2**30:.1f} GiB total, {disk.free / 2**30:.1f} GiB free')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-build-isolation', '-r', str(PROJECT_ROOT / 'requirements-kernels.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-deps', '-r', str(PROJECT_ROOT / 'requirements-accelerator.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-c', 'import coder_sft; print(coder_sft.__version__)'], check=True)
print('Restart the runtime once if Torch or CUDA packages were replaced, then continue below.')

In [ ]:
import os, sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['CODER_SFT_WORKDIR'] = str(WORK_ROOT)
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add a write-enabled HF_TOKEN to Colab Secrets and grant this notebook access.')
os.environ['HF_TOKEN'] = hf_token
from huggingface_hub import HfApi
HF_OWNER = HfApi(token=hf_token).whoami()['name']
DATA_REPO_ID = f'{HF_OWNER}/Qwen3.5-2B-Coder-Data'
MODEL_REPO_ID = f'{HF_OWNER}/Qwen3.5-2B-Coder-SFT'
MODEL_REPO_PRIVATE = True
BASE = str(PROJECT_ROOT / 'configs/base.yaml')
DATA = str(PROJECT_ROOT / 'configs/data_v1.yaml')
HARDWARE = str(PROJECT_ROOT / 'configs/hardware.yaml')
STAGE1 = str(PROJECT_ROOT / 'configs/stage1_8k.yaml')
STAGE2 = str(PROJECT_ROOT / 'configs/stage2_repo_32k.yaml')

## Download the validated data handoff
No source-dataset preparation or repository cloning runs on GPU. The command downloads only normalized JSONL, manifests, and bundle metadata, then validates completeness before training.

In [ ]:
subprocess.run(['sync-data', 'download', '--config', BASE, '--repo-id', DATA_REPO_ID], check=True)
print(f'Downloaded prepared data from: https://huggingface.co/datasets/{DATA_REPO_ID}')

In [ ]:
from coder_sft.config import load_config, write_resolved_config
from coder_sft.hardware import discover_hardware, resolve_hardware
from coder_sft.utils import environment_manifest, write_json
preflight = resolve_hardware(load_config(BASE, HARDWARE, STAGE1), 'stage1', discover_hardware())
REPORTS = WORK_ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
write_resolved_config(preflight, REPORTS / 'preflight_resolved.yaml')
write_json(environment_manifest(), REPORTS / 'preflight_environment.json')
print(preflight['resolved_hardware'])

## Optional one-step smoke test
This uses eight examples from the downloaded full dataset and does not overwrite it. Leave enabled on the first run.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'none', '--limit', '8', '--max-steps', '1', '--output-dir', str(WORK_ROOT / 'outputs/smoke')], check=True)

## Baseline generation and Stage 1
Generated benchmark code is saved only; it is never executed in Colab.

In [ ]:
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'humaneval', '--output', str(REPORTS / 'base_humaneval.jsonl')], check=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'mbpp', '--output', str(REPORTS / 'base_mbpp.jsonl')], check=True)
subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'auto'], check=True)
STAGE1_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s1-8k/final_adapter'
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', str(STAGE1_ADAPTER), '--dataset', 'humaneval', '--output', str(REPORTS / 'stage1_humaneval.jsonl')], check=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', str(STAGE1_ADAPTER), '--dataset', 'mbpp', '--output', str(REPORTS / 'stage1_mbpp.jsonl')], check=True)

## External gate, memory profile, and Stage 2
Score the four generated files with EvalPlus in its Docker container outside Colab. Upload normalized `base.json` and `stage1.json` to `/content/qwen35-2b-coder/reports`, then run the next cell. Stage 2 refuses to run when its quality gate fails.

In [ ]:
BASE_SCORES = REPORTS / 'base.json'
STAGE1_SCORES = REPORTS / 'stage1.json'
if not BASE_SCORES.exists() or not STAGE1_SCORES.exists():
    raise FileNotFoundError('External EvalPlus scores are required: reports/base.json and reports/stage1.json')
subprocess.run(['compare-runs', '--baseline', str(BASE_SCORES), '--candidate', str(STAGE1_SCORES), '--stage', 'stage1', '--output', str(REPORTS / 'stage1_gate.json')], check=True)
subprocess.run(['profile-memory', '--config', BASE, '--config', HARDWARE, '--config', STAGE2], check=True)
subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE2, '--stage', 'stage2', '--adapter', str(STAGE1_ADAPTER), '--resume', 'auto'], check=True)

## Final verified export and model upload
Run after Stage 2 completes. It validates Stage-2 lineage, compares fixed greedy adapter and merged outputs, and uploads the merged BF16 model plus adapter. No W&B account or key is used.

In [ ]:
STAGE2_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s2-repo/final_adapter'
FINAL_EXPORT = WORK_ROOT / 'exports/q35-coder-merged'
publish = ['export-model', '--adapter', str(STAGE2_ADAPTER), '--output', str(FINAL_EXPORT), '--format', 'merged', '--hub-model-id', MODEL_REPO_ID]
if not MODEL_REPO_PRIVATE:
    publish.append('--public')
subprocess.run(publish, check=True)
print(f'Published final model: https://huggingface.co/{MODEL_REPO_ID}')